# Reranker experiments — completing the paper's evidence

Four outstanding items, run together because they share one expensive setup (corpus embedding + retrieval), so the index is built **once** instead of four times.

| | Item | Why |
|---|---|---|
| **A** | Fix `bge-reranker-base` | It scored *below* the no-rerank baseline (CIs fully negative) — the signature of a 2D-score-array bug, not a weak model |
| **B** | Arabic-specific rerankers | `GATE-Reranker-V1` and `Namaa-ARA-Reranker-V1`. GATE's card claims dialect coverage, so it directly tests the asymmetry claim |
| **C** | Rerank depth ablation | top-10/20/50 — turns "we used top-20" into a justified choice |
| **D** | **Asymmetry test** | The paper's novel claim is reranking helps Darija *more* than MSA. Two separate CIs don't establish that. This runs the paired difference-of-differences bootstrap that does. |

**D is the most important cell in this notebook** — it's what makes the central claim demonstrated rather than merely observed.

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`. **GPU required.** Checkpointed.

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "base_encoder": "intfloat/multilingual-e5-base",
    "alpha": 0.8,

    "depths": [10, 20, 50],          # C: rerank depth ablation
    "primary_depth": 20,             # depth used for A, B and D

    "rerankers": [
        "BAAI/bge-reranker-v2-m3",              # current best, for comparison
        "BAAI/bge-reranker-base",               # A: rerun with the scoring guard
        "NAMAA-Space/GATE-Reranker-V1",         # B: Arabic-specific, claims dialect coverage
        "NAMAA-Space/Namaa-ARA-Reranker-V1",    # B: second Arabic-specific option
    ],

    "k_values": (1, 3, 5, 10),
    "bootstrap_n": 1000,
    "seed": 42,
    "batch_size": 16,
    "checkpoint": "reranker_experiments_checkpoint.pkl",
}

### Load data

In [ ]:
import json, re, gc, os, pickle
import numpy as np
import pandas as pd

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]
print(f"Corpus {len(corpus)} | evaluating all {len(qa)} items")

### BM25

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

### Retrieve once at the DEEPEST depth; shallower depths are prefixes

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

MAXD = max(CONFIG["depths"] + [CONFIG["primary_depth"]])
print(f"Building index and retrieving top-{MAXD} (shallower depths are prefixes of this)...")

bi = SentenceTransformer(CONFIG["base_encoder"])
corpus_emb = np.asarray(
    bi.encode([f"passage: {t}" for t in corpus_texts],
              normalize_embeddings=True, batch_size=32, show_progress_bar=True), "float32")

def retrieve(query, k):
    q = bi.encode([f"query: {query}"], normalize_embeddings=True)[0]
    s = (CONFIG["alpha"] * minmax(corpus_emb @ q)
         + (1 - CONFIG["alpha"]) * minmax(np.asarray(bm25.get_scores(tokenize(query)))))
    return [corpus_ids[i] for i in np.argsort(-s)[:k]]

candidates = {}
for field in ["msa_query", "darija_query"]:
    candidates[field] = {q["id"]: retrieve(q[field], MAXD) for q in qa}
    print(f"  {field} done")

del bi, corpus_emb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Evaluation helper

In [ ]:
def evaluate_order(ordered_by_qid):
    out = {f"R@{k}": [] for k in CONFIG["k_values"]}
    rr = []
    for q in qa:
        ordered = ordered_by_qid[q["id"]]
        gold = q["source_chunk_id"]
        pos = ordered.index(gold) + 1 if gold in ordered else None
        for k in CONFIG["k_values"]:
            out[f"R@{k}"].append(1.0 if (pos and pos <= k) else 0.0)
        rr.append(1.0 / pos if pos else 0.0)
    return {**{k: np.array(v) for k, v in out.items()}, "MRR": np.array(rr)}

results = {}
if os.path.exists(CONFIG["checkpoint"]):
    with open(CONFIG["checkpoint"], "rb") as f:
        results.update(pickle.load(f))
    print(f"Resumed {len(results)} cached entries.")

# Baselines: no reranking, at each depth
for depth in CONFIG["depths"]:
    for field in ["msa_query", "darija_query"]:
        key = ("no_rerank", field, depth)
        if key not in results:
            results[key] = evaluate_order(
                {q["id"]: candidates[field][q["id"]][:depth] for q in qa})

print("\nBaseline (no reranking):")
for depth in CONFIG["depths"]:
    m = results[("no_rerank", "darija_query", depth)]
    print(f"  top-{depth:<3} Darija R@1={m['R@1'].mean():.3f}  R@5={m['R@5'].mean():.3f}")

### A + B + C: run every reranker at every depth, with the scoring guard

In [ ]:
from sentence_transformers import CrossEncoder

def score_pairs(ce, query, cand_ids):
    """THE BUG FIX: some cross-encoders return a 2D per-class array instead of
    a 1D relevance score. argsort on 2D sorts within rows, producing garbage
    ordering -- which is what made bge-reranker-base score below baseline."""
    pairs = [(query, corpus_map[c]) for c in cand_ids]
    scores = np.asarray(ce.predict(pairs, batch_size=CONFIG["batch_size"],
                                   show_progress_bar=False))
    if scores.ndim > 1:
        scores = scores[:, -1]   # take the positive/relevant class
    return scores

def run_reranker(model_name):
    short = model_name.split("/")[-1]
    needed = [(f, d) for d in CONFIG["depths"] for f in ["msa_query", "darija_query"]
              if (short, f, d) not in results]
    if not needed:
        print(f"\n=== {short} === (cached, skipping)")
        return

    print(f"\n=== {short} ===")
    try:
        ce = CrossEncoder(model_name, max_length=512, trust_remote_code=True,
                          automodel_args={"torch_dtype": torch.float32})
    except Exception as e:
        print(f"  SKIPPED (load): {type(e).__name__}: {str(e)[:140]}")
        return

    try:
        # Score once at max depth, then slice -- reranking top-10 is the same
        # computation as the first 10 of top-50's candidate list reordered.
        for field in ["msa_query", "darija_query"]:
            full_order = {}
            for q in qa:
                cands = candidates[field][q["id"]]
                scores = score_pairs(ce, q[field], cands)
                full_order[q["id"]] = (cands, scores)

            for depth in CONFIG["depths"]:
                reordered = {}
                for qid, (cands, scores) in full_order.items():
                    sub_c, sub_s = cands[:depth], scores[:depth]
                    order = np.argsort(-sub_s)
                    reordered[qid] = [sub_c[i] for i in order]
                results[(short, field, depth)] = evaluate_order(reordered)
            print(f"  {field} done (all depths)")
    except Exception as e:
        print(f"  SKIPPED (inference): {type(e).__name__}: {str(e)[:140]}")
    finally:
        del ce
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    with open(CONFIG["checkpoint"], "wb") as f:
        pickle.dump(results, f)

for name in CONFIG["rerankers"]:
    run_reranker(name)

### Results table

In [ ]:
rows = []
for (method, field, depth), m in results.items():
    rows.append({"method": method, "query": field, "depth": depth,
                 **{k: float(v.mean()) for k, v in m.items()}})
df = pd.DataFrame(rows)
df.to_csv("reranker_experiments_full.csv", index=False)

D = CONFIG["primary_depth"]
print("=" * 90)
print(f"RESULTS AT TOP-{D}")
print("=" * 90)
main = df[df.depth == D].pivot(index="method", columns="query", values=["R@1", "R@5", "MRR"])
print(main.to_string(float_format=lambda x: f"{x:.3f}"))

### A: did the bug fix change bge-reranker-base?

In [ ]:
print("\n" + "=" * 90)
print("A. BUG FIX CHECK — bge-reranker-base")
print("=" * 90)
prev = {"msa_query": 0.520, "darija_query": 0.365}   # from the buggy run
sub = df[(df.method == "bge-reranker-base") & (df.depth == D)]
if len(sub):
    for _, r in sub.iterrows():
        base = df[(df.method == "no_rerank") & (df["query"] == r["query"]) & (df.depth == D)]["R@1"].iloc[0]
        print(f"  {r['query']:<14} before fix {prev[r['query']]:.3f} | after fix {r['R@1']:.3f} | "
              f"no-rerank baseline {base:.3f}")
    print("\n  If it now scores at or above the baseline, the 2D-score bug explained it.")
    print("  If it is still below, the model is genuinely unsuited and should be")
    print("  reported as such rather than silently dropped.")
else:
    print("  Model did not run.")

### C: depth ablation

In [ ]:
print("\n" + "=" * 90)
print("C. RERANK DEPTH ABLATION (Darija R@1)")
print("=" * 90)
dep = df[df["query"] == "darija_query"].pivot(index="depth", columns="method", values="R@1")
print(dep.to_string(float_format=lambda x: f"{x:.3f}"))
print("""
Deeper reranking raises the ceiling (more chance the gold passage is present)
but costs proportionally more cross-encoder passes. Flat or falling numbers with
depth mean the extra candidates are adding noise, not signal.""")

### D: the asymmetry test (the paper's novel claim)

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])

def boot_ci(d):
    idx = rng.integers(0, len(d), size=(CONFIG["bootstrap_n"], len(d)))
    m = d[idx].mean(axis=1)
    return d.mean(), *np.percentile(m, [2.5, 97.5])

print("\n" + "=" * 90)
print("D. ASYMMETRY TEST — does reranking help Darija MORE than MSA?")
print("=" * 90)
print("Two separate CIs on the two gains do NOT establish that the gains differ.")
print("This is a paired difference-of-differences bootstrap, which does.\n")

asym = []
for method in df.method.unique():
    if method == "no_rerank":
        continue
    try:
        dar_gain = (results[(method, "darija_query", D)]["R@1"]
                    - results[("no_rerank", "darija_query", D)]["R@1"])
        msa_gain = (results[(method, "msa_query", D)]["R@1"]
                    - results[("no_rerank", "msa_query", D)]["R@1"])
    except KeyError:
        continue
    # Paired: both gains are measured on the same question set, so the
    # difference is taken per item before bootstrapping.
    dd, lo, hi = boot_ci(dar_gain - msa_gain)
    verdict = ("helps Darija more" if lo > 0 else
               "helps MSA more" if hi < 0 else "no significant asymmetry")
    asym.append({"method": method,
                 "darija_gain": dar_gain.mean(), "msa_gain": msa_gain.mean(),
                 "difference": dd, "lo": lo, "hi": hi, "verdict": verdict})

asymdf = pd.DataFrame(asym)
print(asymdf.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))
asymdf.to_csv("asymmetry_test.csv", index=False)
print("""
'helps Darija more' with a CI excluding zero is the result the paper's novel
claim requires. 'no significant asymmetry' means the claim must be softened to
a descriptive observation rather than a demonstrated effect.""")

### Dialect gap by method

In [ ]:
print("\n" + "=" * 90)
print(f"DIALECT GAP AT TOP-{D} (MSA - Darija, R@1)")
print("=" * 90)
gaps = []
for method in df.method.unique():
    try:
        d = results[(method, "msa_query", D)]["R@1"] - results[(method, "darija_query", D)]["R@1"]
    except KeyError:
        continue
    g, lo, hi = boot_ci(d)
    gaps.append({"method": method, "gap": g, "lo": lo, "hi": hi,
                 "significant": "yes" if lo > 0 else "no"})
gapdf = pd.DataFrame(gaps).sort_values("gap")
print(gapdf.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))
gapdf.to_csv("gap_by_reranker.csv", index=False)

from google.colab import files
files.download("reranker_experiments_full.csv")
files.download("asymmetry_test.csv")
files.download("gap_by_reranker.csv")